In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 29. Tensor Parallelism and Pipeline Parallelism

> **Learning Goals**
> - Understand how tensor parallelism (TP) partitions matrices
> - Understand the bubble problem in pipeline parallelism (PP)
> - Explain 1F1B scheduling

## 29.1 Why Model Parallelism Is Needed

Data parallelism requires the full model to fit on one GPU. Models with 70B+ parameters do not fit.

Solutions:
- **Tensor Parallelism (TP)**: split weights within a layer across GPUs
- **Pipeline Parallelism (PP)**: split layers across GPUs
- **3D Parallelism**: combine DP + TP + PP, as in GPT-3 training

## 29.2 Tensor Parallelism (Megatron-LM Style)

Partition a linear layer $Y = XW$.

### Column Parallel
$W = [W_1; W_2]$ (split by columns)
$$Y = XW = X[W_1; W_2] = [XW_1, XW_2]$$
- Each GPU computes its own $W_i$
- All-Gather concatenates outputs

### Row Parallel
$W = [W_1, W_2]$ (split by rows)
$$Y = XW = [X_1, X_2][W_1; W_2] = X_1 W_1 + X_2 W_2$$
- Each GPU computes $X_i W_i$
- All-Reduce sums the partial results

Megatron uses column parallelism for attention and a column+row combination for FFN, requiring one All-Reduce.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

# Column Parallel Linear simulation
class ColumnParallelLinear(nn.Module):
    """W split by columns (simplified simulation)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert out_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.out_per_gpu = out_features // n_gpus
        # weights on each GPU
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(in_features, self.out_per_gpu) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # each GPU computes its own partition
        outputs = [x @ w for w in self.weights]
        # All-Gather: concatenate results
        return torch.cat(outputs, dim=-1)

# Row Parallel Linear
class RowParallelLinear(nn.Module):
    """W split by rows (split the input too)."""
    def __init__(self, in_features, out_features, n_gpus=2):
        super().__init__()
        assert in_features % n_gpus == 0
        self.n_gpus = n_gpus
        self.in_per_gpu = in_features // n_gpus
        self.weights = nn.ParameterList([
            nn.Parameter(torch.randn(self.in_per_gpu, out_features) * 0.1)
            for _ in range(n_gpus)
        ])

    def forward(self, x):
        # split the input too
        x_chunks = torch.chunk(x, self.n_gpus, dim=-1)
        # compute on each GPU
        partial = [xc @ w for xc, w in zip(x_chunks, self.weights)]
        # All-Reduce: average
        return sum(partial)

# Test
d_in, d_out = 64, 64
n_gpus = 2

# Original
torch.manual_seed(0)
W_full = torch.randn(d_in, d_out) * 0.1

# Column parallel simulation
col = ColumnParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    # Weight W_fullof splitwith Setting
    for i in range(n_gpus):
        col.weights[i].copy_(W_full[:, i*col.out_per_gpu:(i+1)*col.out_per_gpu])

x = torch.randn(4, d_in)
y_full = x @ W_full
y_col = col(x)
print(f"Column Parallel error: {(y_full - y_col).abs().max():.2e}")

# Row parallel
row = RowParallelLinear(d_in, d_out, n_gpus)
with torch.no_grad():
    for i in range(n_gpus):
        row.weights[i].copy_(W_full[i*row.in_per_gpu:(i+1)*row.in_per_gpu, :])
y_row = row(x)
print(f"Row Parallel Error: {(y_full - y_row).abs().max():.2e}")
print("\n=> after splitting and combining, the result matches the original!")


## 29.3 Splitting Attention Heads

MHA naturally fits tensor parallelism: assign different heads to different GPUs.

- $h$ heads, $n$ GPUs → $h/n$ heads per GPU
- Each head computes its own $W_i^Q, W_i^K, W_i^V$
- Concatenate head outputs and combine them with All-Reduce

## 29.4 Pipeline Parallelism

Split layers across GPUs:
- GPU 0: layers 1-4
- GPU 1: layers 5-8
- GPU 2: layers 9-12

Forward pass goes GPU 0 → 1 → 2. Backpropagation goes in reverse.

### Bubble Problem
While one GPU is working, others may wait, creating idle time (bubbles).

Bubble ratio: $\frac{p-1}{m + p - 1}$
- $p$: num of pipeline stages
- $m$: num of microbatches

Solution: **microbatching** fills the pipeline by splitting a batch into $m$ microbatches.


In [ ]:
# pipeline bubble visualization
def pipeline_bubble_ratio(p, m):
    """Bubble Ratio."""
    return (p - 1) / (m + p - 1)

# for various p and m
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bubble Ratio
ax = axes[0]
for p in [2, 4, 8, 16]:
    ms = np.arange(1, 50)
    ratios = [pipeline_bubble_ratio(p, m) for m in ms]
    ax.plot(ms, ratios, label=f'p={p}', linewidth=2)
ax.set_xlabel('num of microbatches m')
ax.set_ylabel('Bubble Ratio')
ax.set_title('Pipeline Bubble Ratio')
ax.legend(); ax.grid(True, alpha=0.3)

# Gantt chart (simplified)
ax = axes[1]
p, m = 4, 8
# Work schedule for each GPU: Forward Pass F, Backward Pass B, idle .
schedule = []
for stage in range(p):
    s = []
    # forward in stage order
    for micro in range(m):
        # start time = stage + micro
        for t in range(stage + micro + 1):
            pass
    # simplified: backward after forward
    s = ['.'] * (2 * m + 2 * (p - 1))
    # forward Pattern
    for micro in range(m):
        start = stage + micro
        if start < len(s):
            s[start] = f'F{micro}'
    # backward
    for micro in range(m):
        start = m + 2 * (p - 1) - stage + micro
        if 0 <= start < len(s):
            s[start] = f'B{micro}'
    schedule.append(s)

# Visualization (simple)
for i, s in enumerate(schedule):
    for j, x in enumerate(s):
        if x.startswith('F'):
            ax.barh(i, 1, left=j, color='blue', alpha=0.7)
        elif x.startswith('B'):
            ax.barh(i, 1, left=j, color='red', alpha=0.7)
        else:
            ax.barh(i, 1, left=j, color='gray', alpha=0.3)
ax.set_xlabel('Time Step')
ax.set_ylabel('GPU (stage)')
ax.set_title(f'Pipeline schedule (p={p}, m={m})')
ax.set_yticks(range(p))
plt.tight_layout()
plt.savefig('../figures/ch29_pipeline_bubble.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"p=4, m=8 bubble ratio: {pipeline_bubble_ratio(4, 8)*100:.1f}%")
print(f"p=4, m=32 Bubble Ratio: {pipeline_bubble_ratio(4, 32)*100:.1f}%")
print("=> more microbatches reduce the bubble.")


## 29.5 1F1B Scheduling

A simple schedule runs all forward passes and then all backward passes, which uses a lot of activation memory.

**1F1B** alternates one forward pass and one backward pass, saving activation memory.


## 29.6 3D Parallelism (DP + TP + PP)

GPT-3 (175B) training used:
- DP: data parallelism (split batches)
- TP: split attention/FFN weights
- PP: split layers

Example: 1024 GPUs = 8 DP × 8 TP × 16 PP

## 29.7 Key Takeaways

| Parallelism | Split Unit | Communication | Memory |
|---|---|---|---|
| Data (DP) | data | All-Reduce gradients | full model |
| Tensor (TP) | weight matrices | All-Reduce / All-Gather | part of a layer |
| Pipeline (PP) | layers | point-to-point | part of layers + activations |
| 3D | all of the above | mixed | minimum |

## Exercises

1. Implement column parallel and row parallel layers and show they match the original result.
2. Compute pipeline bubble ratios for p=2, 4, 8 and m=4, 16, 64.
3. Simulate tensor parallelism by splitting attention heads across 4 GPUs.
4. Explain why 1024 GPUs might be arranged as DP=8, TP=8, PP=16 in 3D parallelism.
5. Explain how 1F1B scheduling saves memory.

> Solutions: `solutions/ch29_solutions.ipynb`
